# Particles and orbit from one SPAM run

Runs one model with `sim.simulate`, then plots the particles as a 2D histogram with the secondary galaxy's
orbit on top. The plotting follows `plot_2d_histogram` from `IllustrisTNG/tng_functions.py` on
`working_research`: `log1p` counts, with an outlined orbit line.

**To run it** (from this repo's root, in bash or PowerShell), then open the link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
The primary galaxy stays at the origin in SPAM's frame, so the orbit is the secondary's path relative to it.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patheffects import withStroke

from galactic_tidal_constraints.simr import sim

## Settings
Change these and rerun. The model is SDSS 587722984435351614, from JSPAM's `archive/data.txt`.

In [ ]:
MODEL = [
    -9.93853, -4.5805, 15.43348, -1.54025, -2.97899, 4.69234, 34.41671, 29.49607, 2.97913, 4.2501, 301.0,
    35.5, 310.80745, 321.98757, 0.3, 0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
]  # fmt: skip
N_PARTICLES = 4000  # in each disk
SEED = 1
BINS = 200
ENDINGS = (0.0, 0.25, 0.5, 0.75, 1.0, 1.2)  # fractions of the run; 1.0 is the normal end

KPC_PER_UNIT = 15.0  # one SPAM distance unit

## Run it
One integration saves the particles at every ending. The orbit run takes the same endings, so it covers the
same time.

In [ ]:
endings = sim.Endings("fraction", ENDINGS)
particles = sim.simulate(
    MODEL, sim.SimulatorConfig(n1=N_PARTICLES, n2=N_PARTICLES, endings=endings), seed=SEED
)
orbit = sim.simulate(MODEL, sim.SimulatorConfig(kind="orbit", endings=endings))

print(f"particles: {particles.snapshots.shape} (endings, particles + secondary's center, x y z vx vy vz)")
print(f"orbit:     {orbit.orbit.shape} (steps, x y z vx vy vz)")
print(f"start: t = {particles.t_start:.3f} = {sim.to_gyr(particles.t_start):.3f} Gyr")
print(f"start-time fallback: {particles.start_fallback}")
print("endings (Gyr):", np.round(sim.to_gyr(particles.times), 3))

## The plot function
- **Color:** one hue, light to dark, for particles per bin. Empty bins match the background.
- **`log1p`:** tidal features have far fewer particles per bin than the galaxy centers, so they'd disappear on
  a linear scale.
- **Orbit:** orange, with a white outline so it stays visible over dark bins.

In [ ]:
DENSITY = LinearSegmentedColormap.from_list(
    "density", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
ORBIT_COLOR = "#eb6834"
INK = "#0b0b0b"
AXIS_NAMES = "xyz"


def histogram(points, limits, axes=(0, 1), bins=BINS):
    "log1p of particles per bin. `points` has one row per particle."
    counts, _, _ = np.histogram2d(points[:, axes[0]], points[:, axes[1]], bins=bins, range=limits)
    return np.log1p(counts)


def square_limits(*point_sets, axes=(0, 1), padding=1.05):
    "Equal, square limits centered on everything given, so distances look the same in both directions."
    points = np.vstack([p[:, list(axes)] for p in point_sets])
    low, high = points.min(axis=0), points.max(axis=0)
    center, half = (low + high) / 2, (high - low).max() / 2 * padding
    return [(center[0] - half, center[0] + half), (center[1] - half, center[1] + half)]


def plot_particles(ax, snapshot, orbit_path=None, limits=None, axes=(0, 1), vmax=None):
    """Draw one snapshot as a 2D histogram, with the orbit on top.

    `snapshot` is one ending, shape (particles + 1, 6), with the secondary's center last.
    `orbit_path` is the orbit's positions up to this time. Returns the image, for a color bar.
    """
    disks, center = snapshot[:-1], snapshot[-1]
    if limits is None:
        limits = square_limits(snapshot, *([] if orbit_path is None else [orbit_path]), axes=axes)

    image = ax.imshow(
        histogram(disks, limits, axes).T,
        origin="lower",
        extent=[*limits[0], *limits[1]],
        cmap=DENSITY,
        vmin=0,
        vmax=vmax,
        interpolation="nearest",
    )
    if orbit_path is not None:
        outline = withStroke(linewidth=4, foreground="white")
        x, y = orbit_path[:, axes[0]], orbit_path[:, axes[1]]
        ax.plot(x, y, color=ORBIT_COLOR, linewidth=2, label="Secondary's orbit", path_effects=[outline])
    marker = {"color": INK, "markersize": 8, "markeredgecolor": "white", "linestyle": "none"}
    ax.plot(0, 0, marker="o", label="Primary's center", **marker)
    ax.plot(center[axes[0]], center[axes[1]], marker="s", label="Secondary's center", **marker)

    ax.set_xlim(limits[0])
    ax.set_ylim(limits[1])
    ax.set_xlabel(f"{AXIS_NAMES[axes[0]]} (1 unit = {KPC_PER_UNIT:.0f} kpc)")
    ax.set_ylabel(f"{AXIS_NAMES[axes[1]]} (1 unit = {KPC_PER_UNIT:.0f} kpc)")
    ax.tick_params(colors="#52514e", labelsize=9)
    return image

## The normal end of the run
`ENDINGS.index(1.0)` picks the ending at the run's normal end, about t = 0.

In [ ]:
final = particles.snapshots[ENDINGS.index(1.0)]
step = particles.steps[ENDINGS.index(1.0)]
path = orbit.orbit[: step + 1, :3]

fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")
image = plot_particles(ax, final, path)
ax.set_title(
    f"Particles and orbit at t = {sim.to_gyr(particles.times[ENDINGS.index(1.0)]):.3f} Gyr", loc="left"
)
ax.legend(loc="best", frameon=True)
fig.colorbar(image, ax=ax, label="log(1 + particles per bin)", shrink=0.8)
plt.show()

## Three views
The orbit is 3D, so one view can hide structure. Each panel has its own scale and limits.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5.2), layout="constrained")
for ax, axes in zip(axs, [(0, 1), (0, 2), (1, 2)]):
    image = plot_particles(ax, final, path, axes=axes)
    ax.set_title(f"{AXIS_NAMES[axes[0]]}-{AXIS_NAMES[axes[1]]} view", loc="left")
axs[0].legend(loc="best", frameon=True, fontsize=8)
plt.show()

## Many endings
One simulation, saved at each entry in `ENDINGS`. All panels share their limits and color scale, so they can
be compared directly. The orbit is drawn up to each panel's time. Each panel's limits cover the whole run, so
close-in structure is small here; the zoomed plot below shows it better.

In [ ]:
limits = square_limits(orbit.orbit, *particles.snapshots)
vmax = max(histogram(s[:-1], limits).max() for s in particles.snapshots)

fig, axs = plt.subplots(2, 3, figsize=(14, 9), layout="constrained", sharex=True, sharey=True)
for ax, snapshot, step, time in zip(axs.flat, particles.snapshots, particles.steps, particles.times):
    image = plot_particles(ax, snapshot, orbit.orbit[: step + 1, :3], limits=limits, vmax=vmax)
    ax.set_title(f"t = {sim.to_gyr(time):.2f} Gyr ({step} steps)", loc="left")
for ax in axs[0]:
    ax.set_xlabel("")
for ax in axs[:, 1:].flat:
    ax.set_ylabel("")
axs[0, 0].legend(loc="lower left", frameon=True, fontsize=8)
fig.colorbar(image, ax=axs, label="log(1 + particles per bin)", shrink=0.6)
plt.show()

## Zoomed in on each galaxy
The same final particles, on a tighter window around each center, with the same limits width and color scale.

In [ ]:
HALF_WIDTH = 6.0
center = final[-1]
windows = {
    "Primary": [(-HALF_WIDTH, HALF_WIDTH), (-HALF_WIDTH, HALF_WIDTH)],
    "Secondary": [
        (center[0] - HALF_WIDTH, center[0] + HALF_WIDTH),
        (center[1] - HALF_WIDTH, center[1] + HALF_WIDTH),
    ],
}
vmax = max(histogram(final[:-1], w).max() for w in windows.values())

fig, axs = plt.subplots(1, 2, figsize=(12, 5.5), layout="constrained")
for ax, (name, window) in zip(axs, windows.items()):
    image = plot_particles(ax, final, path, limits=window, vmax=vmax)
    ax.set_title(name, loc="left")
axs[0].legend(loc="best", frameon=True, fontsize=8)
fig.colorbar(image, ax=axs, label="log(1 + particles per bin)", shrink=0.8)
plt.show()